In [4]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.ensemble import RandomForestRegressor
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

df = fetch_openml("house_prices", as_frame=True, parser="auto").frame

# category dtype ko object bana do (imputer ke saath safe rehta hai)
for c in df.select_dtypes(include="category").columns:
    df[c] = df[c].astype(object)

print("Shape:", df.shape)
display(df.head(10))

missing = df.isnull().sum()
print("\nMissing values (only columns with missing):")
print(missing[missing > 0].sort_values(ascending=False))

print("\nSalePrice summary:")
print(df["SalePrice"].describe())

Shape: (1460, 81)


,Id,MSSubClass,MSZoning,LotFrontage,LotArea,Street,Alley,LotShape,LandContour,Utilities,...,PoolArea,PoolQC,Fence,MiscFeature,MiscVal,MoSold,YrSold,SaleType,SaleCondition,SalePrice
0,1,60,RL,65.0,8450,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,2,2008,WD,Normal,208500
1,2,20,RL,80.0,9600,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,5,2007,WD,Normal,181500
2,3,60,RL,68.0,11250,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,9,2008,WD,Normal,223500
3,4,70,RL,60.0,9550,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,2,2006,WD,Abnorml,140000
4,5,60,RL,84.0,14260,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,12,2008,WD,Normal,250000
5,6,50,RL,85.0,14115,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,MnPrv,Shed,700,10,2009,WD,Normal,143000
6,7,20,RL,75.0,10084,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,8,2007,WD,Normal,307000
7,8,60,RL,NaN,10382,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,Shed,350,11,2009,WD,Normal,200000
8,9,50,RM,51.0,6120,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,4,2008,WD,Abnorml,129900
9,10,190,RL,50.0,7420,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,1,2008,WD,Normal,118000



Missing values (only columns with missing):
PoolQC          1453
MiscFeature     1406
Alley           1369
Fence           1179
FireplaceQu      690
LotFrontage      259
GarageCond        81
GarageType        81
GarageYrBlt       81
GarageFinish      81
GarageQual        81
BsmtFinType2      38
BsmtExposure      38
BsmtCond          37
BsmtQual          37
BsmtFinType1      37
MasVnrType         8
MasVnrArea         8
Electrical         1
dtype: int64

SalePrice summary:
count      1460.000000
mean     180921.195890
std       79442.502883
min       34900.000000
25%      129975.000000
50%      163000.000000
75%      214000.000000
max      755000.000000
Name: SalePrice, dtype: float64


In [5]:
y = df["SalePrice"]
X = df.drop(columns=["SalePrice", "Id"], errors="ignore")   # Id identifier hai
print("X:", X.shape, " y:", y.shape)

X: (1460, 79)  y: (1460,)


In [6]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
print("Train:", X_train.shape, "Test:", X_test.shape)

Train: (1168, 79) Test: (292, 79)


In [7]:
num_cols = X_train.select_dtypes(include="number").columns
cat_cols = X_train.select_dtypes(exclude="number").columns

preprocess = ColumnTransformer([
    ("num", SimpleImputer(strategy="median"), num_cols),
    ("cat", Pipeline([
        ("imp", SimpleImputer(strategy="constant", fill_value="None")),
        ("ohe", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
    ]), cat_cols),
])

X_train_p = preprocess.fit_transform(X_train)  
X_test_p  = preprocess.transform(X_test)

feature_names = preprocess.get_feature_names_out()
feature_names = [n.split("__", 1)[1] for n in feature_names]
print("Features after preprocessing:", X_train_p.shape[1])

Features after preprocessing: 301


In [8]:
def evaluate(model, Xt, yt):
    p = model.predict(Xt)
    mse = mean_squared_error(yt, p)
    return {"MAE": mean_absolute_error(yt, p), "MSE": mse,
            "RMSE": np.sqrt(mse), "R2": r2_score(yt, p)}

rf_default = RandomForestRegressor(random_state=42)
rf_default.fit(X_train_p, y_train)
default_scores = evaluate(rf_default, X_test_p, y_test)
print(default_scores)

{'MAE': 17714.937534246576, 'MSE': 851835413.825739, 'RMSE': np.float64(29186.219587773594), 'R2': 0.888944043471376}


In [ ]:
param_grid = {
    "n_estimators":      [100, 200],
    "max_depth":         [None, 10, 20],
    "min_samples_split": [2, 5],
    "min_samples_leaf":  [1, 2],
    "max_features":      [0.7, 1.0],
}

grid = GridSearchCV(
    RandomForestRegressor(random_state=42),
    param_grid,
    cv=5,
    scoring="neg_mean_squared_error",
    n_jobs=-1,
    verbose=1,
)
grid.fit(X_train_p, y_train)

best_mse = -grid.best_score_           
best_rmse = np.sqrt(best_mse)

print("Best hyperparameters:", grid.best_params_)
print("Best CV score (neg MSE):", grid.best_score_)
print(f"Best CV MSE : {best_mse:,.2f}")
print(f"Best CV RMSE: {best_rmse:,.2f}")

Fitting 5 folds for each of 48 candidates, totalling 240 fits


In [ ]:
best_rf = grid.best_estimator_
tuned_scores = evaluate(best_rf, X_test_p, y_test)

comparison = pd.DataFrame({
    "Default RF": default_scores,
    "Tuned RF (GridSearchCV)": tuned_scores,
}).T
comparison["Improved?"] = [
    "-", "-"
]
comparison = pd.DataFrame({"Metric": list(default_scores.keys()),
                           "Default RF": list(default_scores.values()),
                           "Tuned RF": list(tuned_scores.values())})
display(comparison.round(4))

In [ ]:
y_pred = best_rf.predict(X_test_p)

plt.figure(figsize=(7, 6))
plt.scatter(y_test, y_pred, alpha=0.6, edgecolor="k", color="steelblue")
lims = [min(y_test.min(), y_pred.min()), max(y_test.max(), y_pred.max())]
plt.plot(lims, lims, "r--", label="Perfect prediction")
plt.xlabel("Actual SalePrice")
plt.ylabel("Predicted SalePrice")
plt.title("Actual vs Predicted House Prices (Tuned RF)")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

In [ ]:
imp = pd.Series(best_rf.feature_importances_, index=feature_names).sort_values(ascending=False)
top5 = imp.head(5)
print(top5.round(4))

plt.figure(figsize=(7, 4))
top5[::-1].plot(kind="barh", color="seagreen")
plt.xlabel("Importance")
plt.title("Top 5 Most Influential Features")
plt.tight_layout()
plt.show()